In [ ]:
!pip install -q PyMuPDF spacy transformers pandas torch
!python -m spacy download en_core_web_sm

import torch, transformers, pandas as pd, spacy

print("Torch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("Pandas :", pd.__version__)
print("GPU actif :", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 50.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Torch : 2.9.0+cu126
Transformers : 5.0.0
Pandas : 2.2.2
GPU actif : True
Device utilisé : cuda


In [ ]:
import os

TRAIN_DIR = "/content/drive/MyDrive/data/train"
TEST_DIR  = "/content/drive/MyDrive/data/test"

print("Train PDFs :", os.listdir(TRAIN_DIR))
print("Test PDFs  :", os.listdir(TEST_DIR))


Train PDFs : ['Gorodetski2022.pdf', 'Brigo2022.pdf', 'Huemer2022.pdf', 'Chien2022.pdf', 'Choi2022.pdf', 'Edward2022.pdf', 'GuoH2022.pdf', 'Cheng_F2022.pdf', 'Campagnini2022.pdf', 'Cheng_X2022.pdf', 'Gao2022.pdf', 'Andrew2022.pdf', 'AlvarezRodriguez2022.pdf', 'Habets2022.pdf', 'Hassan2022.pdf', 'Hurst2022.pdf', 'Almberg2022.pdf', 'Guo2022.pdf', 'Feng2022.pdf', 'vanDerLuebe222.pdf', 'Wong2022.pdf', 'Merk2022.pdf', 'Xie2022.pdf', 'Maciag2022.pdf', 'Roller2022.pdf', 'Lu2022.pdf', 'Verboven2022.pdf', 'Liu2022.pdf', 'Xiao2022.pdf', 'Kuno2022.pdf', 'Oliveira2022.pdf', 'Lai2022.pdf', 'Li2022.pdf', 'Kuo2022.pdf', 'Juraev2022.pdf', 'Zhao2022.pdf', 'Singh2022.pdf', 'Turan2022.pdf', 'Ritter2022.pdf', 'Reverberi2022.pdf', 'OConnell2022.pdf', 'Shen2022.pdf', 'Yan2023.pdf', 'Tokita2022.pdf', 'Kats2022.pdf', 'Saller2022.pdf', 'Yu2022.pdf', 'Stenwig2022.pdf', 'Redjdal2022.pdf', 'Kleppe2022.pdf', 'Pal2022.pdf', 'Reinecke2022.pdf', 'Pedrosa2022.pdf', 'Schamberg2022.pdf', 'Sabharwal2022.pdf', 'Shapiro2022

TAPE 1 — Prétraitement des articles

Dans cette étape, nous chargeons les articles scientifiques au format PDF contenus dans le dossier train/ et nous en extrayons le texte brut.

L’objectif est de transformer des documents PDF, qui ne sont pas directement exploitables par un modèle de langage, en texte lisible par un programme.
Pour cela, nous utilisons la bibliothèque PyMuPDF, qui permet de parcourir chaque page d’un PDF et d’en extraire le contenu textuel.

Chaque article est ensuite converti en une chaîne de caractères contenant l’ensemble de son texte.
Ces textes serviront de base pour les étapes suivantes, notamment la segmentation en phrases et la création du jeu de données d’entraînement pour le modèle BioBERT.

In [ ]:
import fitz

def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text
train_texts = []

for f in os.listdir(TRAIN_DIR):
    if f.lower().endswith(".pdf"):
        train_texts.append(extract_text_from_pdf(os.path.join(TRAIN_DIR, f)))

print("Nombre de PDF train traités :", len(train_texts))


Nombre de PDF train traités : 56


## Pourquoi segmenter le texte en phrases ?

Les modèles de type BERT, et en particulier BioBERT, imposent une limite maximale de **512 tokens** en entrée.  
Un article scientifique complet ou même un paragraphe long dépasse largement cette contrainte et ne peut donc pas être traité directement par le modèle.

La segmentation du texte en phrases permet :
- de respecter la contrainte de longueur imposée par le modèle,
- d’obtenir des unités sémantiques courtes et cohérentes,
- de faciliter l’identification d’informations précises telles que les métriques d’évaluation (accuracy, Dice, AUC, etc.),
- d’augmenter la taille du jeu de données en générant plusieurs exemples à partir d’un même document.

Cette étape est donc essentielle pour garantir un entraînement efficace et pertinent du modèle BioBERT.


La bibliothèque spaCy est utilisée pour segmenter automatiquement le texte , garantit une détection fiable des frontières de phrases, contrairement à une segmentation naïve basée uniquement sur la ponctuation.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm", disable=["ner", "tagger"])

train_sentences = []

for text in train_texts:
    doc = nlp(text)
    for sent in doc.sents:
        s = sent.text.strip()
        if 25 < len(s) < 300:
            train_sentences.append(s)

print("Nombre de phrases extraites :", len(train_sentences))


/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Nombre de phrases extraites : 17296


Je commence par extraire le texte brut des articles au format PDF.
Pour réduire le bruit et optimiser le temps de calcul, je ne traite que la section “Results / Evaluation”, car c’est là que les métriques d’évaluation sont généralement rapportées.

In [ ]:
START_KEYS = ["RESULTS", "EVALUATION", "PERFORMANCE"]
END_KEYS   = ["DISCUSSION", "CONCLUSION", "REFERENCES"]

def extract_results_section(text):
    t = text.upper()
    start = min([t.find(k) for k in START_KEYS if k in t], default=-1)
    if start == -1:
        return ""
    end = min([t.find(k, start+1) for k in END_KEYS if k in t[start+1:]], default=-1)
    return text[start:end] if end != -1 else text[start:]
results_sentences = []

for text in train_texts:
    results_text = extract_results_section(text)
    if not results_text.strip():
        continue
    for sent in nlp(results_text).sents:
        s = sent.text.strip()
        if 25 < len(s) < 300:
            results_sentences.append(s)

print("Phrases issues des Results :", len(results_sentences))



Phrases issues des Results : 2954


In [ ]:
METRIC_MAP = {
    "accuracy":"ACC","acc":"ACC",
    "auc":"AUC-ROC","roc":"AUC-ROC","auc-roc":"AUC-ROC",
    "f1":"F1","f1-score":"F1",
    "dice":"DICE","iou":"IOU",
    "rmse":"RMSE","mae":"MAE","mse":"MSE",
    "sensitivity":"SE","se":"SE",
    "specificity":"SP","sp":"SP",
    "mcc":"MCC","ppv":"PPV","npv":"NPV","iqr":"IQR"
}


def label_sentence(s):
    s = s.lower()
    return 1 if any(k in s for k in METRIC_MAP) else 0

df_train = pd.DataFrame({"sentence": results_sentences})
df_train["label"] = df_train["sentence"].apply(label_sentence)

df_train["label"].value_counts()



,count
label,
1,2083
0,871


In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
class MetricDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.df.sentence[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.df.label[idx])
        }

train_dataset = MetricDataset(df_train)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
).to(device)

model



pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were ne

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
from transformers import Trainer, TrainingArguments

#définir tous les paramètres d’entraînement
training_args = TrainingArguments(
    output_dir="./biobert_metric", # Dossier où les checkpoints (modèle sauvegardé) seront enregistrés.
    per_device_train_batch_size=16, #Taille du batch GPU : modèle apprend sur 16 phrases en même temps avant de mettre à jour les poids
    num_train_epochs=15, #Le modèle va passer 15 fois sur tout le dataset d’entraînement.
    learning_rate=2e-5, # Taux d’apprentissage.
    fp16=torch.cuda.is_available(),
    logging_steps=50, #Affiche les logs (loss, learning rate, etc.) toutes les 50 étapes d’entraînement.
    save_strategy="epoch", #Sauvegarde un checkpoint à chaque fin d’époque.
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

trainer.train() #Lancement de l’entraînement


Step,Training Loss
50,0.585071
100,0.551165
150,0.572788
200,0.501045
250,0.423681
300,0.416673
350,0.401031
400,0.306288
450,0.206995
500,0.229820


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2775, training_loss=0.0933227773690764, metrics={'train_runtime': 868.7576, 'train_samples_per_second': 51.004, 'train_steps_per_second': 3.194, 'total_flos': 2914612715750400.0, 'train_loss': 0.0933227773690764, 'epoch': 15.0})

In [ ]:
import re
from collections import defaultdict

METRIC_REGEX = re.compile(
    r"\b(" + "|".join(METRIC_MAP.keys()) + r")\b",
    re.IGNORECASE
)

pdf_metrics = defaultdict(set)
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
with torch.no_grad():
    for f in os.listdir(TEST_DIR):
        if not f.endswith(".pdf"):
            continue

        text = extract_text_from_pdf(os.path.join(TEST_DIR, f))
        for sent in nlp(text).sents:
            s = sent.text.strip()
            if not (25 < len(s) < 300):
                continue

            inputs = tokenizer(
                s, return_tensors="pt",
                truncation=True, max_length=128
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            prob = torch.softmax(model(**inputs).logits, dim=1)[0,1].item()

            if prob > 0.8:
                for m in METRIC_REGEX.findall(s.lower()):
                    pdf_metrics[f].add(METRIC_MAP[m])


/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [ ]:
final_df = pd.DataFrame([
    {"pdf": k, "metrics": ", ".join(sorted(v))}
    for k, v in pdf_metrics.items()
])

final_df



,pdf,metrics
0,Freitas2022.pdf,"ACC, AUC-ROC, F1, SE, SP"
1,Dros2022.pdf,"AUC-ROC, SE, SP"
2,Baharlouei2022.pdf,ACC
3,Islam2022.pdf,"ACC, AUC-ROC, F1, MCC, SE, SP"
4,Davis2022.pdf,"ACC, AUC-ROC, SE"
5,Duanmu2022.pdf,"ACC, AUC-ROC, SP"
6,Huyut2022.pdf,"ACC, SE"
7,Arnal2022.pdf,"ACC, AUC-ROC, PPV"
8,Durán2022.pdf,"ACC, SE, SP"
9,Kaplan2022.pdf,"ACC, AUC-ROC, F1, MCC, SE"


In [ ]:
output_path = "/content/drive/MyDrive/metrics_by_pdf.csv"
final_df.to_csv(output_path, index=False)
print("✅ CSV généré :", output_path)


✅ CSV généré : /content/drive/MyDrive/metrics_by_pdf.csv
